In [13]:
import pandas as pd
import numpy as np
from helpers import get_feature_importance
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from joblib import dump, load

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, classification_report
from category_encoders import CountEncoder
from category_encoders.target_encoder import TargetEncoder
seed = 42

## Тюнинг гиперпараметров моделей

Для повышения качества моделей был выполнен поэтапный тюнинг гиперпараметров.

На первом этапе использовался RandomizedSearchCV с широкими диапазонами значений гиперпараметров. Это позволило эффективно исследовать пространство параметров и определить области, в которых модели показывают наилучшие результаты, без чрезмерных вычислительных затрат.

На втором этапе диапазоны значений были сужены вокруг найденных оптимальных областей, после чего применён GridSearchCV. Такой подход позволил более точно подобрать оптимальные значения гиперпараметров.

Таким образом, комбинированная стратегия RandomizedSearchCV → GridSearchCV позволила сначала глобально исследовать пространство параметров, а затем выполнить точную настройку моделей.


In [14]:
df = pd.read_csv('bank.csv')
result_cross_val_baseline = load('result_cross_val_baseline.joblib')
result_test_baseline = load('result_test_baseline.joblib')

In [15]:
class QuantileClipper(BaseEstimator, TransformerMixin):
        def __init__(self, lower=0.01, upper=0.99):
            self.lower = lower
            self.upper = upper
            
        def fit(self, X, y=None):
            self.lower_bounds_ = np.quantile(X, self.lower, axis=0)
            self.upper_bounds_ = np.quantile(X, self.upper, axis=0)
            return self
        
        def transform(self, X):
            return np.clip(X, self.lower_bounds_, self.upper_bounds_)

        def get_feature_names_out(self, input_features=None):
            return input_features
    
# ====== 1. Разделение на X и y ======
X = df.drop('deposit', axis=1)
y = df['deposit'].map({'yes': 1, 'no': 0})  # бинаризация таргета

# ====== 2. Определение типов признаков ======
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

# ====== 3. Препроцессинг ======
num_pipeline = Pipeline([
    ('clipper', QuantileClipper(lower=0.01, upper=0.99)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])  

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# ====== 4. Разбиение с stratify ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=seed,
    stratify=y  # разбиенте со стратификацией
)

# ====== 5. Модели ======
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(random_state=seed),
    'XGBoost': XGBClassifier(
        eval_metric='logloss',
        random_state=seed
    )
}

In [17]:
# ====== ТЮНИНГ ГИПЕРПАРАМЕТРОВ ======================== 

param_rs_grids = {
    'LogisticRegression': {
        'model__C': [0.01, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 2, 5, 10],
        'model__class_weight': [None, 'balanced']
    },
    'RandomForest': {
        'model__n_estimators': [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750 , 800, 850, 900, 950, 1000],
        'model__max_depth': [2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, None],
        'model__min_samples_leaf': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
        'model__max_features': ['sqrt', 'log2']
    },
    'XGBoost': {
        'model__n_estimators': [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750 , 800, 850, 900, 950, 1000],
        'model__max_depth': [2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
        'model__learning_rate': [0.005, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2],
        'model__subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
        'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
        'model__min_child_weight': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) # разбиенте со стратификацией

best_models_rs = {}
best_params_rs = {}

for name, model in models.items():
    print(f"\nTuning {name}...")
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    random_search = RandomizedSearchCV(
        estimator = pipeline,
        param_distributions = param_rs_grids[name],
        n_iter = 100,
        cv=cv,
        scoring='recall',   # главная метрика 
        n_jobs=-1,
        verbose=1
    )
    
    random_search.fit(X_train, y_train)
    
    best_models_rs[name] = random_search.best_estimator_
    best_params_rs[name] = random_search.best_params_

# Таблица лучших параметров
best_params_rs_df = pd.DataFrame(best_params_rs)


Tuning LogisticRegression...
Fitting 5 folds for each of 58 candidates, totalling 290 fits


d:\Programs\Python31210\Projects\bank\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 58 is smaller than n_iter=100. Running 58 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Tuning RandomForest...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Tuning XGBoost...
Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [18]:
results_tuned_rs = {}

for name, best_model in best_models_rs.items():
    
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    
    roc = roc_auc_score(y_test, y_pred_proba)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results_tuned_rs[name] = [roc, accuracy, precision, recall, f1]

result_test_tuned_rs_df = pd.DataFrame(
    results_tuned_rs,
    index=['roc_auc', 'accuracy', 'precision', 'recall', 'f1'] ).T

print("\nBaseline CV results:")
display(result_cross_val_baseline)

print("\nBest hyperparameters:")
display(best_params_rs_df.fillna('-'))

print("\nTest results after tuning:")
display(result_test_tuned_rs_df)

print("\nCompare baseline and tuned:")
display(result_test_tuned_rs_df-result_test_baseline)


Baseline CV results:


,roc_auc,accuracy,precision,recall,f1
LogisticRegression,0.9016 ± 0.0043,0.8238 ± 0.0074,0.8274 ± 0.0055,0.7939 ± 0.0188,0.8102 ± 0.0097
RandomForest,0.9184 ± 0.0045,0.85 ± 0.0073,0.8203 ± 0.0052,0.8752 ± 0.0126,0.8468 ± 0.0081
XGBoost,0.9203 ± 0.0052,0.8513 ± 0.0053,0.8263 ± 0.0047,0.8688 ± 0.0133,0.847 ± 0.0063



Best hyperparameters:


,LogisticRegression,RandomForest,XGBoost
model__class_weight,balanced,-,-
model__C,0.13,-,-
model__n_estimators,-,800,100.0
model__min_samples_leaf,-,2,-
model__max_features,-,sqrt,-
model__max_depth,-,18,17.0
model__subsample,-,-,0.7
model__min_child_weight,-,-,1.0
model__learning_rate,-,-,0.08
model__colsample_bytree,-,-,0.8



Test results after tuning:


,roc_auc,accuracy,precision,recall,f1
LogisticRegression,0.907206,0.828034,0.817925,0.819471,0.818697
RandomForest,0.922035,0.860278,0.824913,0.895085,0.858568
XGBoost,0.926562,0.865652,0.837790,0.888469,0.862385



Compare baseline and tuned:


,roc_auc,accuracy,precision,recall,f1
LogisticRegression,-0.000091,0.002687,-0.009526,0.021739,0.006377
RandomForest,0.002707,-0.001343,-0.006797,0.007561,-0.000143
XGBoost,0.000835,0.001343,-0.003222,0.008507,0.002339


In [19]:
# ====== ТЮНИНГ ГИПЕРПАРАМЕТРОВ GridSearchCV =============

param_gridsearch = {
    'LogisticRegression': {
        'model__C': [0.01, 0.02,  0.03,  0.04, 0.05, 0.06, 0.07, 0.075, 0.08, 0.085, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 2, 5, 10],
        'model__class_weight': [None, 'balanced']
    },
    'RandomForest': {
        'model__n_estimators': [750, 800, 850],
        'model__max_depth': [17, 18, 19, None],
        'model__min_samples_leaf': [1, 2, 3],
        'model__max_features': ['sqrt', 'log2']
    },
    'XGBoost': {
        'model__n_estimators': [100, 150, 200],
        'model__max_depth': [16, 17, 18],
        'model__learning_rate': [0.07, 0.08, 0.09],
        'model__subsample': [0.6, 0.7, 0.8],
        'model__colsample_bytree': [0.7, 0.8, 0.9],
        'model__min_child_weight': [1, 2, 3]
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) # разбиенте со стратификацией

best_models_gs = {}
best_params_gs = {}

for name, model in models.items():
    print(f"\nTuning {name}...")
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    grid_search = GridSearchCV(
        estimator = pipeline,
        param_grid = param_gridsearch[name],
        cv=cv,
        scoring='recall',   # главная метрика
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    best_models_gs[name] = grid_search.best_estimator_
    best_params_gs[name] = grid_search.best_params_

# Таблица лучших параметров
best_params_gs_df = pd.DataFrame(best_params_gs)


Tuning LogisticRegression...
Fitting 5 folds for each of 68 candidates, totalling 340 fits

Tuning RandomForest...
Fitting 5 folds for each of 72 candidates, totalling 360 fits

Tuning XGBoost...
Fitting 5 folds for each of 729 candidates, totalling 3645 fits


In [20]:
results_tuned_gs = {}

for name, best_model in best_models_gs.items():
    
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    
    roc = roc_auc_score(y_test, y_pred_proba)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results_tuned_gs[name] = [roc, accuracy, precision, recall, f1]

result_test_tuned_gs_df = pd.DataFrame(
    results_tuned_gs,
    index=['roc_auc', 'accuracy', 'precision', 'recall', 'f1'] ).T

print("\nBaseline CV results:")
display(result_cross_val_baseline)

print("\nBest hyperparameters:")
display(best_params_gs_df.fillna('-'))

print("\nTest results after tuning:")
display(result_test_tuned_gs_df)

print("\nCompare baseline and tuned:")
display(result_test_tuned_gs_df-result_test_baseline)


Baseline CV results:


,roc_auc,accuracy,precision,recall,f1
LogisticRegression,0.9016 ± 0.0043,0.8238 ± 0.0074,0.8274 ± 0.0055,0.7939 ± 0.0188,0.8102 ± 0.0097
RandomForest,0.9184 ± 0.0045,0.85 ± 0.0073,0.8203 ± 0.0052,0.8752 ± 0.0126,0.8468 ± 0.0081
XGBoost,0.9203 ± 0.0052,0.8513 ± 0.0053,0.8263 ± 0.0047,0.8688 ± 0.0133,0.847 ± 0.0063



Best hyperparameters:


,LogisticRegression,RandomForest,XGBoost
model__C,0.13,-,-
model__class_weight,balanced,-,-
model__max_depth,-,-,16.0
model__max_features,-,sqrt,-
model__min_samples_leaf,-,2,-
model__n_estimators,-,800,150.0
model__colsample_bytree,-,-,0.8
model__learning_rate,-,-,0.08
model__min_child_weight,-,-,1.0
model__subsample,-,-,0.8



Test results after tuning:


,roc_auc,accuracy,precision,recall,f1
LogisticRegression,0.907206,0.828034,0.817925,0.819471,0.818697
RandomForest,0.921671,0.861621,0.824805,0.898866,0.860244
XGBoost,0.924908,0.859830,0.832293,0.881853,0.856356



Compare baseline and tuned:


,roc_auc,accuracy,precision,recall,f1
LogisticRegression,-0.000091,0.002687,-0.009526,0.021739,0.006377
RandomForest,0.002342,0.000000,-0.006905,0.011342,0.001534
XGBoost,-0.000819,-0.004478,-0.008719,0.001890,-0.003690


## Итог
Тюнинг гиперпараметров дал небольшой "буст" по метрике RECALL моделей: 

LogisticRegression +2.2%

RandomForest +1.1%

XGBoost +0.2%